# Eye disease classification - Kaggle training

Do these steps in the Kaggle UI before running the notebook:

1. **Settings -> Phone verification** (unlocks GPU and internet).
2. **Add Data** -> attach the `eye-data` dataset.
3. **Settings -> Accelerator** -> GPU **T4** or **P100**.
4. **Settings -> Internet** -> **On** (needed for `git clone` and pretrained weights).

Upload the corrected `eye_data.zip`: its class folders contain no square brackets (`Central Serous Chorioretinopathy Color Fundus`). The training code resolves the local bracketed name automatically.

Do **not** install torch or torchvision; Kaggle already has them. Step 6 does not need `imagehash`.

In [ ]:
!git clone https://github.com/nirjon001/eye-disease-classification.git

In [ ]:
%cd eye-disease-classification

In [ ]:
# Check the real folder name before setting EYE_DATA_ROOT.
!ls /kaggle/input
!ls /kaggle/input/eye-data

In [ ]:
import os
os.environ["EYE_DATA_ROOT"] = "/kaggle/input/eye-data/eye_data"   # check the real path with !ls /kaggle/input
os.environ["EYE_OUTPUT_DIR"] = "/kaggle/working/outputs"
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Quick test first

Two epochs on 200 images. Fix any error here, where it is cheap.

In [ ]:
!python scripts/06_train_cnn.py --model resnet18 --quick

## Full runs

Run **one model at a time** so the first result comes quickly. Uncomment one command, then use **Save Version -> Save & Run All (Commit)**. Commit runs in the background and keeps the outputs even if you close the browser. Repeat for each model listed below.

Every epoch writes the best weights and history CSV, so a stopped session does not lose work.

In [ ]:
!python scripts/06_train_cnn.py --model resnet18 --epochs 50
# !python scripts/06_train_cnn.py --model mobilenet_v2 --epochs 50
# !python scripts/06_train_cnn.py --model simple_cnn --epochs 50
# !python scripts/06_train_cnn.py --model alexnet --epochs 50
# !python scripts/06_train_cnn.py --model resnet50 --epochs 50
# !python scripts/06_train_cnn.py --model efficientnet_b0 --epochs 50
# !python scripts/06_train_cnn.py --model densenet121 --epochs 50
# !python scripts/06_train_cnn.py --model convnext_tiny --epochs 50
# !python scripts/06_train_cnn.py --model swin_t --epochs 50

## Or: run all models at once, in two groups

Models run **one after another**, never in parallel, so GPU memory is safe. Light models go first, then heavy models. Both flags are True below, so one Commit runs everything. If you want the light results quickly, set `RUN_HEAVY = False`, Commit, then set it back to True for a second Commit.

In [ ]:
import subprocess
import sys

LIGHT_MODELS = [
    "simple_cnn",
    "mobilenet_v2",
    "resnet18",
    "efficientnet_b0",
]

HEAVY_MODELS = [
    "alexnet",
    "densenet121",
    "resnet50",
    "convnext_tiny",
    "swin_t",
]

RUN_LIGHT = True
RUN_HEAVY = True
EPOCHS = 50


def train_group(models, group_name):
    for model_name in models:
        print(f"\n===== {group_name}: TRAINING {model_name} =====", flush=True)
        command = [
            sys.executable,
            "scripts/06_train_cnn.py",
            "--model", model_name,
            "--epochs", str(EPOCHS),
        ]
        finished = subprocess.run(command, check=False)
        print(f"===== {group_name}: DONE {model_name}: exit {finished.returncode} =====", flush=True)


if RUN_LIGHT:
    train_group(LIGHT_MODELS, "LIGHT")
if RUN_HEAVY:
    train_group(HEAVY_MODELS, "HEAVY")

## Download results

Everything in `/kaggle/working` appears under the notebook's **Output** tab. Download the `*_best.pth` models, `*_history.csv` files, and `*_curves.png` plots into your local `outputs/` folder. Step 7 (test evaluation) and Step 8 (comparison) can then run locally.